In [4]:
import warnings
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
from tqdm.notebook import tqdm
from gridstatus import CAISO

warnings.filterwarnings('ignore')

In [5]:
START = date(2019, 1, 1)
END   = date(2024, 12, 31)
OUTPUT = '../../data/raw/caiso_hourly.csv'

In [6]:
import os
os.makedirs('../../data/raw', exist_ok=True)

caiso = CAISO()

all_frames = []
cur = START.replace(day=1)
months = []
while cur <= END:
    nxt = cur + relativedelta(months=1)
    months.append((cur.isoformat(), nxt.isoformat()))
    cur = nxt

for ms, me in tqdm(months, desc='Fetching hourly load'):
    try:
        df = caiso.get_load_hourly(start=ms, end=me)
        ca_total = df[df['TAC Area Name'] == 'CA ISO-TAC'][['Interval Start', 'Load']].copy()
        ca_total.columns = ['datetime', 'load_mw']
        all_frames.append(ca_total)
    except Exception as e:
        print(f'  {ms}: {e}')

hourly = pd.concat(all_frames, ignore_index=True)
hourly = hourly.sort_values('datetime').drop_duplicates('datetime').reset_index(drop=True)

hourly.to_csv(OUTPUT, index=False)

Fetching hourly load:   0%|          | 0/72 [00:00<?, ?it/s]

2026-06-10 22:02:30 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'SLD_FCST', 'version': 1}, 'params': {'market_run_id': ['7DA', '2DA', 'DAM', 'ACTUAL', 'RTM'], 'execution_type': [None, 'RTPD', 'RTD']}}
2026-06-10 22:02:30 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6&queryname=SLD_FCST&version=1&market_run_id=ACTUAL&startdatetime=20190101T08:00-0000&enddatetime=20190201T08:00-0000
2026-06-10 22:02:32 - DEBUG - Found 1 files: ['20190101_20190201_SLD_FCST_ACTUAL_20260610_20_02_31_v1.csv']
2026-06-10 22:02:32 - DEBUG - Parsing file: 20190101_20190201_SLD_FCST_ACTUAL_20260610_20_02_31_v1.csv
2026-06-10 22:02:36 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'SLD_FCST', 'version': 1}, 'params': {'market_run_id': ['7DA', '2DA', 'DAM', 'ACTUAL', 'RTM'], 'execution_type': [None, 'RTPD', 'RTD']}}
2026-06-10 22:02:36 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/Sing

KeyboardInterrupt: 